# Retail Customer Intelligence Workshop
## Od danych z Marketplace do AI-powered analizy klientów

**Scenariusz:** Jesteś analitykiem danych w firmie e-commerce specjalizującej się w elektronice
użytkowej. Zarząd chce lepiej rozumieć klientów: kto kupuje, co kupuje i jak przewidzieć
przyszłe zachowania zakupowe. Dane pobierasz z **Databricks Marketplace** — gotowy,
zweryfikowany dataset symulowanych klientów B2B i ich zamówień.

Co zbudujesz w tym warsztacie:
1. Pobierzesz dane z Marketplace i poznasz je (eksploracja SQL)
2. Przeanalizujesz trendy i wzorce zakupowe klientów (zaawansowana analityka SQL)
3. Wykorzystasz AI do generowania insightów biznesowych (AI Functions)
4. Przygotowujesz dane do modelowania — join tabel, parsowanie JSON, RFM (PySpark)
5. Zapiszesz wyniki jako tabelę Gold w Delta Lake
6. Zbudujesz model predykcyjny segmentacji klientów (ML + MLflow)
7. Wygenerujesz prognozę przychodów
8. Udostępnisz wyniki biznesowi (Dashboard + Genie Space)

**Funkcjonalności Databricks, które zobaczysz:**
- **Databricks Marketplace** — pozyskiwanie gotowych datasetów
- **Unity Catalog** — 3-poziomowa hierarchia (catalog.schema.table)
- **SQL + PySpark** — wielojęzyczny notebook
- **AI Functions** — ai_query, ai_classify bezpośrednio w SQL
- **Delta Lake** — ACID, Time Travel, wersjonowanie
- **MLflow** — tracking eksperymentów, model registry
- **Dashboard + Genie Space** — udostępnianie wyników biznesowi

> **Uwaga:** Ten notebook to **część 1 z 4**. W kolejnych warsztatach:
> WS2 — zabezpieczymy dane PII (guardrails, monitoring, ewaluacja),
> WS3 — zbudujemy chatbota RAG i Knowledge Assistant na dokumentach PDF,
> WS4 — stworzymy pełną aplikację agentorową (UC Functions → Agent → Databricks App).

## Sekcja 1: Dane z Marketplace i eksploracja
**Funkcjonalności:** Databricks Marketplace, Unity Catalog (3-poziomowa hierarchia), SQL w notebooku, wizualizacje inline

Dane do tego warsztatu pochodzą z **Databricks Marketplace** — wbudowanego katalogu
gotowych datasetów, które można od razu użyć w swoim workspace.

**Jak pobraliśmy dane:**
1. Otwórz **Marketplace** w panelu bocznym Databricks
2. Wyszukaj `Databricks Simulated Retail Customer Data`
3. Kliknij **Get instant access** — dataset pojawia się jako nowy katalog w Unity Catalog

Po pobraniu mamy katalog `databricks_simulated_retail_customer_data` z trzema tabelami:
- `customers` — 28,813 klientów B2B (dane adresowe, segment lojalnościowy)
- `sales` — 360 rekordów sprzedaży z kategoriami produktów
- `sales_orders` — 4,074 zamówienia z danymi JSON (produkty, promocje, kliknięcia)

Zaczynamy od poznania tych danych. Używamy SQL bezpośrednio w notebooku — Databricks
obsługuje wiele języków w jednym notebooku (Python, SQL, Markdown).

### Sekcja 1b: Zanim klikniesz „Get instant access” — przegląd warunków licencji

Dataset z Marketplace to **produkt danych dostawcy**, nie nasze dane. Zanim trafi do
pipeline'u, Compliance Officer chce wiedzieć **na jakich warunkach** go używamy.
W listingu Marketplace sprawdź i **zanotuj** (to zapis do audytu, a nie formalność):

| Co sprawdzić | Gdzie w listingu | Dlaczego |
| --- | --- | --- |
| **Dostawca** i typ produktu | Nagłówek listingu, sekcja *Provider* | Kto odpowiada za jakość i aktualizacje danych |
| **Licencja / Terms of use** | Sekcja *Terms* lub link *License* (czytaj **aktualną** treść — nie zakładaj typu licencji) | Czy wolno użyć danych komercyjnie, czy wolno je udostępniać dalej |
| **Zakres i aktualizacja** | *Data product details* — tabele, opis, częstotliwość odświeżania | Czy dane są snapshotem czy strumieniem; jak często zmienią się nasze wyniki |
| **PII** | Opis kolumn (np. `tax_id`, `customer_name`) | Nawet dane **symulowane** traktujemy jak PII — to ćwiczy nawyki na WS2 |

**Zapis przeglądu licencji (uzupełnij podczas warsztatu):**

| Pole | Wartość |
| --- | --- |
| Produkt | `Databricks Simulated Retail Customer Data` |
| Dostawca | Databricks (dataset demonstracyjny) |
| Warunki | *(wpisz po przeczytaniu listingu — nie zgaduj)* |
| Data przeglądu / kto zaakceptował | *(dzisiejsza data, Twój e-mail)* |

> **Dlaczego to ważne:** katalog z Marketplace jest **read-only** (Delta Sharing). Wszystko, co zbudujemy
> (`gold_customer_360`), to **nasza kopia pochodna** — i to ona podlega naszym zasadom governance z WS2.
> Poniższa komórka pokazuje, że Unity Catalog „wie”, skąd pochodzą dane.

In [0]:
-- Katalog z Marketplace to katalog typu Delta Sharing — Unity Catalog przechowuje nazwę dostawcy i share'a.
-- Zwróć uwagę na pola: Catalog Type, Provider Name, Share Name, Owner, Created.
DESCRIBE CATALOG EXTENDED databricks_simulated_retail_customer_data;

info_name,info_value
Catalog Name,databricks_simulated_retail_customer_data
Comment,
Owner,katarzyna.palach@cloudsonmars.com
Catalog Type,Delta Sharing
Created By,katarzyna.palach@cloudsonmars.com
Created At,2026-09-01 AD at 10:18:48 UTC
Updated By,katarzyna.palach@cloudsonmars.com
Updated At,2026-09-08 AD at 08:34:54 UTC


In [0]:
-- Lista tabel udostępnionych w share — to jest dokładny zakres „produktu danych”, który zaakceptowaliśmy.
-- Zapisz go w rekordzie przeglądu licencji powyżej.
SHOW TABLES IN databricks_simulated_retail_customer_data.v01;

database,tableName,isTemporary
v01,customers,false
v01,sales,false
v01,sales_orders,false


In [0]:
-- Ile mamy danych? Jaki zakres?
-- Zwróć uwagę na adresowanie: catalog.schema.table (Unity Catalog)
-- Dane z Marketplace są w katalogu: databricks_simulated_retail_customer_data

SELECT 'customers' AS tabela, COUNT(*) AS wiersze, COUNT(DISTINCT customer_id) AS unikalne_id
FROM databricks_simulated_retail_customer_data.v01.customers
UNION ALL
SELECT 'sales_orders', COUNT(*), COUNT(DISTINCT customer_id)
FROM databricks_simulated_retail_customer_data.v01.sales_orders
UNION ALL
SELECT 'sales', COUNT(*), COUNT(DISTINCT customer_id)
FROM databricks_simulated_retail_customer_data.v01.sales

tabela,wiersze,unikalne_id
customers,28813,28670
sales_orders,4074,1942
sales,360,25


In [0]:
-- Rozkład klientów per segment lojalności i stan
-- loyalty_segment: 0 = nowy, 1 = okazjonalny, 2 = regularny, 3 = VIP
-- Po uruchomieniu: kliknij "+" przy wyniku → wybierz wykres słupkowy

SELECT 
  loyalty_segment,
  state,
  COUNT(*) AS liczba_klientow,
  ROUND(AVG(units_purchased), 1) AS avg_units_purchased,
  ROUND(AVG(lat), 4) AS avg_lat
FROM databricks_simulated_retail_customer_data.v01.customers
WHERE state IN ('NY', 'CA', 'FL', 'OH', 'MA')  -- Top 5 stanów
GROUP BY loyalty_segment, state
ORDER BY loyalty_segment, liczba_klientow DESC

loyalty_segment,state,liczba_klientow,avg_units_purchased,avg_lat
0,NY,1325,2.6,42.0224
0,CA,1111,2.5,35.4487
0,FL,995,2.6,27.963
0,OH,754,2.6,40.4093
0,MA,675,2.6,42.2546
1,NY,450,6.5,42.0051
1,CA,398,6.5,35.238
1,FL,329,6.5,28.1822
1,OH,274,6.5,40.4759
1,MA,253,6.6,42.2788


## Sekcja 2: Zaawansowana analityka SQL
**Funkcjonalności:** CTE (Common Table Expressions), Window Functions, LAG, RANK, analiza kohortowa

Teraz przechodzimy do bardziej zaawansowanych zapytań — łączymy tabele klientów
z zamówieniami, obliczamy trendy tygodniowe i budujemy ranking klientów.

Kluczowe koncepty SQL, które zobaczymy:
- **CTE** (Common Table Expressions) — modularyzacja zapytań
- **JOIN** — łączenie tabel customers → sales_orders
- **Window Functions** — LAG, RANK, PARTITION BY
- **Aggregacje czasowe** — trend tygodniowy zamówień

In [0]:
-- Trend tygodniowy zamówień z growth rate (Week-over-Week)
-- Pokazuje: CTE, Window Functions, LAG, TRY_DIVIDE

WITH weekly_orders AS (
  SELECT 
    DATE_TRUNC('week', from_unixtime(order_datetime)) AS week,
    COUNT(*) AS total_orders,
    SUM(number_of_line_items) AS total_items,
    COUNT(DISTINCT customer_id) AS unique_customers
  FROM databricks_simulated_retail_customer_data.v01.sales_orders
  GROUP BY 1
),
with_growth AS (
  SELECT *,
    LAG(total_orders) OVER (ORDER BY week) AS prev_week_orders,
    ROUND(TRY_DIVIDE(
      total_orders - LAG(total_orders) OVER (ORDER BY week),
      LAG(total_orders) OVER (ORDER BY week)
    ) * 100, 1) AS growth_pct
  FROM weekly_orders
)
SELECT * FROM with_growth
ORDER BY week

-- Tip: Kliknij "+" przy wyniku → wykres liniowy, X=week, Y=total_orders

week,total_orders,total_items,unique_customers,prev_week_orders,growth_pct
null,45,90,32,null,null
2019-07-29T00:00:00.000Z,134,296,100,45,197.8
2019-08-05T00:00:00.000Z,257,536,148,134,91.8
2019-08-12T00:00:00.000Z,263,526,164,257,2.3
2019-08-19T00:00:00.000Z,280,588,169,263,6.5
2019-08-26T00:00:00.000Z,260,527,183,280,-7.1
2019-09-02T00:00:00.000Z,242,486,163,260,-6.9
2019-09-09T00:00:00.000Z,240,475,150,242,-0.8
2019-09-16T00:00:00.000Z,275,547,145,240,14.6
2019-09-23T00:00:00.000Z,265,548,163,275,-3.6


In [0]:
-- Top klienci per segment lojalności (RANK, PARTITION BY)
-- Pokazuje: Window Functions do rankingów, JOIN dwóch tabel

WITH customer_orders AS (
  SELECT 
    c.customer_id,
    c.customer_name,
    c.state,
    c.loyalty_segment,
    COUNT(o.order_number) AS total_orders,
    SUM(o.number_of_line_items) AS total_items
  FROM databricks_simulated_retail_customer_data.v01.customers c
  INNER JOIN databricks_simulated_retail_customer_data.v01.sales_orders o
    ON c.customer_id = o.customer_id
  GROUP BY c.customer_id, c.customer_name, c.state, c.loyalty_segment
)
SELECT * FROM (
  SELECT 
    customer_name,
    state,
    loyalty_segment,
    total_orders,
    total_items,
    RANK() OVER (PARTITION BY loyalty_segment ORDER BY total_orders DESC) AS rank_in_segment
  FROM customer_orders
)
WHERE rank_in_segment <= 3
ORDER BY loyalty_segment, rank_in_segment

customer_name,state,loyalty_segment,total_orders,total_items,rank_in_segment
"DIAZ, EDUARDO",GA,0,2,3,1
population health consultants,AZ,0,2,4,1
"CURTIN, MICHAEL J",NY,0,2,3,1
"ESPINOZA, FERNANDO",FL,0,2,2,1
prairie crossing charter school district 900,PA,0,2,2,1
art to wear,MT,1,2,5,1
company of rock house,TX,1,2,3,1
hudson river industries,SD,1,2,3,1
"BROOKS, PHILLIP",LA,2,2,3,1
"KRAFT, VICKI M",MA,2,2,2,1


## Sekcja 3: AI Functions — sztuczna inteligencja w SQL
**Funkcjonalności:** `ai_query()`, `ai_classify()` — LLM bezpośrednio w zapytaniach SQL

To jedna z najciekawszych możliwości Databricks — możesz wywołać model językowy (LLM) jako zwykłą
funkcję SQL! Nie potrzebujesz Pythona, API keys, ani infrastruktury — wszystko
działa od razu, bez dodatkowej konfiguracji.

W kontekście naszych danych klientów użyjemy AI do:
- **Generowania insightów** o segmentach klientów
- **Klasyfikacji** klientów na podstawie ich zachowań zakupowych

In [0]:
-- AI generuje biznesowy insight o segmentach klientów
-- Pokazuje: ai_query() — wywołanie LLM w SQL

WITH segment_stats AS (
  SELECT
    loyalty_segment,
    COUNT(*) AS customers,
    ROUND(AVG(units_purchased), 1) AS avg_purchases,
    ROUND(AVG(units_purchased) * COUNT(*), 0) AS estimated_total_units
  FROM databricks_simulated_retail_customer_data.v01.customers
  GROUP BY loyalty_segment
)
SELECT
  loyalty_segment,
  customers,
  avg_purchases,
  ai_query(
    'databricks-meta-llama-3-3-70b-instruct',
    CONCAT(
      'Jesteś analitykiem retail. Segment lojalności ', CAST(loyalty_segment AS STRING),
      ' ma ', CAST(customers AS STRING), ' klientów, ',
      'średnia zakupów: ', CAST(avg_purchases AS STRING), ' sztuk. ',
      'Segment 0=nowy, 1=okazjonalny, 2=regularny, 3=VIP. ',
      'Podaj JEDNĄ krótką rekomendację biznesową (max 20 słów) po polsku.'
    )
  ) AS ai_recommendation
FROM segment_stats
ORDER BY loyalty_segment

loyalty_segment,customers,avg_purchases,ai_recommendation
0,11097,2.5,Zwiększ frekwencję zakupów w segmencie lojalności 0 poprzez kampanie promocyjne.
1,3883,6.5,Zwiększ częstotliwość zakupów w segmencie lojalności 1.
2,4292,8.7,"Zwiększ zaangażowanie klientów z segmentu 2, oferując im premiowe rabaty."
3,9541,29.9,Zwiększ zaangażowanie klientów VIP ofertami personalizowanymi.


In [0]:
-- AI klasyfikuje typ klienta na podstawie zachowań
-- Pokazuje: ai_classify() — kategoryzacja danych przez AI
-- Optymalizacja: najpierw agregujemy, potem wywołujemy AI tylko na 10 wierszach

WITH top_customers AS (
  SELECT
    c.customer_name,
    c.state,
    c.units_purchased,
    c.loyalty_segment,
    COUNT(o.order_number) AS total_orders
  FROM databricks_simulated_retail_customer_data.v01.customers c
  LEFT JOIN databricks_simulated_retail_customer_data.v01.sales_orders o
    ON c.customer_id = o.customer_id
  WHERE c.state IN ('NY', 'CA')
  GROUP BY c.customer_name, c.state, c.units_purchased, c.loyalty_segment
  ORDER BY c.units_purchased DESC
  LIMIT 10
)
SELECT
  customer_name,
  state,
  units_purchased,
  loyalty_segment,
  total_orders,
  ai_classify(
    CONCAT(
      'Customer with ', CAST(units_purchased AS STRING), ' total units purchased, ',
      CAST(total_orders AS STRING), ' orders placed, ',
      'loyalty segment ', CAST(loyalty_segment AS STRING), ' (0=new, 3=VIP)'
    ),
    ARRAY('high_value_loyal', 'growing_potential', 'at_risk_churning', 'dormant')
  ) AS customer_type
FROM top_customers

customer_name,state,units_purchased,loyalty_segment,total_orders,customer_type
helios electronics limited,NY,814,3,54,high_value_loyal
mct digital,CA,784,3,52,high_value_loyal
epi-electrochemical products inc,NY,780,3,58,high_value_loyal
modern digital imaging,CA,769,3,58,high_value_loyal
rpm optoelectronics,NY,762,3,63,high_value_loyal
digital photo solutions ltd,CA,742,3,62,high_value_loyal
"unified digital group, llc",CA,729,3,50,high_value_loyal
"digital products international, inc.",NY,722,3,50,high_value_loyal
"otbda , outside the box digital agency ,",NY,699,3,53,high_value_loyal
"digital blue, inc. (db)",CA,622,3,50,high_value_loyal


## Sekcja 4: PySpark — Cleaning, Join i Feature Engineering
**Funkcjonalności:** DataFrame API, JSON parsing, JOIN, Window Functions, wielojęzyczność notebooka

Przechodzimy na Python! W tej sekcji:
1. **Łączymy tabele** — customers + sales_orders (JOIN)
2. **Parsujemy JSON** — wyciągamy dane o produktach i promocjach z kolumn tekstowych
3. **Budujemy cechy klienta** — RFM (Recency, Frequency, Monetary), cechy geograficzne
4. **Przygotowujemy dane do ML** — predykcja `loyalty_segment`

> **Wielojęzyczność:** Zwróć uwagę, że SQL używaliśmy wyżej, a teraz
> przechodzimy na Python w tym samym notebooku — Databricks obsługuje oba.

In [0]:
%python
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

# ---- Krok 1: Wczytanie danych z Marketplace (Unity Catalog) ----
catalog = "databricks_simulated_retail_customer_data"
customers = spark.table(f"{catalog}.v01.customers")
orders = spark.table(f"{catalog}.v01.sales_orders")

print(f"Customers: {customers.count():,} rows")
print(f"Orders: {orders.count():,} rows")

# ---- Krok 2: Parsowanie JSON z ordered_products ----
# Kolumna ordered_products to string z JSON array — wyciągamy łączną wartość zamówienia
from pyspark.sql.functions import from_json, schema_of_json, explode, col

# Definiujemy schemat JSON (każdy produkt ma curr, id, name, price, qty, unit)
product_schema = "ARRAY<STRUCT<curr:STRING, id:STRING, name:STRING, price:STRING, qty:STRING, unit:STRING>>"

orders_parsed = (
    orders
    .withColumn("order_date", F.from_unixtime("order_datetime").cast("date"))
    .withColumn("products", F.from_json("ordered_products", product_schema))
    .withColumn("product", F.explode("products"))
    .withColumn("item_price", F.col("product.price").cast("double"))
    .withColumn("item_qty", F.col("product.qty").cast("int"))
    .withColumn("item_revenue", F.col("item_price") * F.col("item_qty"))
    .withColumn("has_promo", F.when(F.col("promo_info") != "[]", 1).otherwise(0))
)

print(f"\nParsed order items: {orders_parsed.count():,}")
orders_parsed.select("customer_id", "order_date", "product.name", "item_price", "item_qty", "item_revenue", "has_promo").show(5, truncate=40)

# ---- Krok 3: Agregacja per klient (RFM + dodatkowe cechy) ----
reference_date = F.lit("2019-11-15").cast("date")  # data referencyjna (koniec danych)

customer_features = (
    orders_parsed
    .groupBy("customer_id")
    .agg(
        F.datediff(reference_date, F.max("order_date")).alias("recency_days"),
        F.count("order_number").alias("frequency"),  # ile itemów zamówiono
        F.countDistinct("order_number").alias("num_orders"),
        F.round(F.sum("item_revenue"), 2).alias("monetary"),
        F.round(F.avg("item_revenue"), 2).alias("avg_item_value"),
        F.sum("has_promo").alias("promo_orders"),
        F.min("order_date").alias("first_order_date"),
        F.max("order_date").alias("last_order_date"),
    )
)

# ---- Krok 4: JOIN z tabelą customers ----
gold_df = (
    customers
    .join(customer_features, "customer_id", "left")
    .select(
        "customer_id",
        "customer_name",
        "tax_id",
        "state",
        "city",
        "loyalty_segment",
        "units_purchased",
        F.col("lat"),
        F.col("lon"),
        F.coalesce("recency_days", F.lit(999)).alias("recency_days"),
        F.coalesce("frequency", F.lit(0)).alias("frequency"),
        F.coalesce("num_orders", F.lit(0)).alias("num_orders"),
        F.coalesce("monetary", F.lit(0.0)).alias("monetary"),
        F.coalesce("avg_item_value", F.lit(0.0)).alias("avg_item_value"),
        F.coalesce("promo_orders", F.lit(0)).alias("promo_orders"),
        "first_order_date",
        "last_order_date",
    )
    # Dodatkowe cechy pochodne
    .withColumn("has_orders", F.when(F.col("num_orders") > 0, 1).otherwise(0))
    .withColumn("promo_ratio", F.round(
        F.when(F.col("frequency") > 0, F.col("promo_orders") / F.col("frequency")).otherwise(0.0), 3))
)

print(f"\nGold table: {gold_df.count():,} rows, {len(gold_df.columns)} columns")
print(f"Columns: {gold_df.columns}")
gold_df.show(5, truncate=30)

Customers: 28,813 rows
Orders: 4,074 rows

Parsed order items: 8,137
+-----------+----------+----------------------------------------+----------+--------+------------+---------+
|customer_id|order_date|                                    name|item_price|item_qty|item_revenue|has_promo|
+-----------+----------+----------------------------------------+----------+--------+------------+---------+
|   19476252|2019-08-01|Rony LBT-GPX555 Mini-System with Blue...|     993.0|       3|      2979.0|        0|
|   19476252|2019-08-01|Aeon 71.5 x 130.9 16:9 Fixed Frame Pr...|     218.0|       3|       654.0|        0|
|   19476252|2019-08-01|Cyber-shot DSC-WX220 Digital Camera (...|     448.0|       2|       896.0|        0|
|    4401099|2019-08-01|Details About Mogitech G920 Xbox Driv...|     293.0|       4|      1172.0|        0|
|   14939501|2019-08-01|Adventura SH 140 II Shoulder Bag (Black)|      27.0|       1|        27.0|        1|
+-----------+----------+-----------------------------------

## Sekcja 5: Delta Lake — zapis i wersjonowanie
**Funkcjonalności:** Delta Lake, saveAsTable, Time Travel, DESCRIBE HISTORY

Zapisujemy przetworzoną tabelę `gold_customer_360` jako tabelę Delta w Unity Catalog.
To będzie nasza **główna tabela** — używana w ML, dashboardach, i w **części 2 warsztatu**
(guardrails, monitoring, ewaluacja).

Delta Lake daje nam:
- **ACID transactions** — niezawodność zapisu
- **Time Travel** — dostęp do poprzednich wersji danych
- **Schema enforcement** — ochrona przed błędnymi danymi

> **Ważne:** Ta tabela zawiera dane PII klientów (customer_name, tax_id, adresy).
> W części 2 warsztatu zabezpieczymy ją przez column masks i row filters.

In [0]:
%python
# Zapis jako tabela Delta w Unity Catalog (warstwa Gold)
# Tabela gold_customer_360 — fundament obu warsztatów

GOLD_TABLE = "workspace.default.gold_customer_360"

(gold_df
    .write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"Tabela zapisana: {GOLD_TABLE}")
print(f"  Wiersze: {spark.table(GOLD_TABLE).count():,}")
print(f"  Kolumny: {len(spark.table(GOLD_TABLE).columns)}")

Tabela zapisana: workspace.default.gold_customer_360
  Wiersze: 28,813
  Kolumny: 19


In [0]:
-- Time Travel — historia zmian tabeli
-- Każdy zapis tworzy nową wersję — możemy wrócić do dowolnej!

DESCRIBE HISTORY workspace.default.gold_customer_360

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
4,2026-09-08T10:59:31.000Z,142316075702328,katarzyna.palach@cloudsonmars.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3153402175013847),567ccd00-8822-4f24-9330-3c28a22e3474,0908-105751-62qtdbr3-v2n,3,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1024054, numDeletionVectorsRemoved -> 0, numOutputRows -> 28813, numOutputBytes -> 1024062)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
3,2026-09-07T13:08:16.000Z,142316075702328,katarzyna.palach@cloudsonmars.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(3153402175013847),d7c054ac-40da-491a-bca6-bb0c8470adea,0907-130640-p4ic4r8p-v2n,2,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1024061, numDeletionVectorsRemoved -> 0, numOutputRows -> 28813, numOutputBytes -> 1024054)",null,Databricks-Runtime/19.6.x-photon-scala2.13
2,2026-09-01T13:51:10.000Z,142316075702328,katarzyna.palach@cloudsonmars.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1132767115158657),e79c6d20-a495-41d1-81ae-76a7011c4143,0901-103053-qaz4j1y-v2n,1,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1024061, numDeletionVectorsRemoved -> 0, numOutputRows -> 28813, numOutputBytes -> 1024061)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
1,2026-09-01T11:29:43.000Z,142316075702328,katarzyna.palach@cloudsonmars.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(1132767115158657),d7f58a92-7448-4822-8af8-f29faa77a524,0901-103053-qaz4j1y-v2n,0,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 1, numRemovedBytes -> 1024061, numDeletionVectorsRemoved -> 0, numOutputRows -> 28813, numOutputBytes -> 1024061)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13
0,2026-09-01T10:54:00.000Z,142316075702328,katarzyna.palach@cloudsonmars.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(2544689905798374),ecd014a0-0f1f-4e6d-92de-401a79d843a1,0901-103053-qaz4j1y-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 28813, numOutputBytes -> 1024061)",null,Databricks-Runtime/19.5.x-aarch64-photon-scala2.13


In [0]:
-- Odczyt danych z poprzedniej wersji (gdybyśmy mieli więcej wersji)
-- SELECT * FROM workspace.default.gold_customer_360 VERSION AS OF 0 LIMIT 5

-- Sprawdźmy schemat naszej tabeli Gold — zwróć uwagę na kolumny PII!
DESCRIBE TABLE workspace.default.gold_customer_360

col_name,data_type,comment
customer_id,bigint,null
customer_name,string,null
tax_id,string,null
state,string,null
city,string,null
loyalty_segment,bigint,null
units_purchased,bigint,null
lat,double,null
lon,double,null
recency_days,int,null


## Sekcja 5b: Tool Calling — UC Function jako narzędzie LLM
**Funkcjonalności:** UC Functions, Foundation Model API, tool calling (function calling)

W sekcji 3 widzieliśmy `ai_query()` i `ai_classify()` — LLM odpowiada na pytanie **w SQL**.
Ale co jeśli chcemy, żeby **LLM sam zdecydował jaką funkcję wywołać**? To jest **tool calling**:

```
Użytkownik: „Jaki jest łączny przychód od VIPów w NY?”
    ↓
LLM analizuje pytanie → wybiera narzędzie: get_revenue_summary(segment=3, state_filter='NY')
    ↓
UC Function wykonuje SQL na gold_customer_360 → zwraca wynik
    ↓
LLM formatuje odpowiedź dla użytkownika
```

**Kluczowa różnica:**
- `ai_query()` — **my** wywołujemy LLM z SQL, podając dane
- **tool calling** — LLM **sam** wywołuje nasze funkcje, gdy uzna to za potrzebne

To fundamentalny mechanizm agentów AI — w Warsztacie 4 zbudujemy pełnego agenta, który używa wielu narzędzi jednocześnie.

In [0]:
-- UC Function zwracająca podsumowanie przychodu per segment i stan
-- Agent (LLM) wywoła tę funkcję automatycznie na podstawie pytania użytkownika

CREATE OR REPLACE FUNCTION workspace.default.get_revenue_summary(
  segment BIGINT COMMENT 'Loyalty segment ID: 0=new/inactive, 1=occasional, 2=regular, 3=VIP. Pass -1 for all segments.',
  state_filter STRING COMMENT 'US state abbreviation (e.g. NY, CA) or ALL for all states.'
)
RETURNS STRING
COMMENT 'Returns revenue summary from gold_customer_360: total revenue, avg per customer, customer count, total orders. Use to answer business questions about revenue by segment and geography.'
RETURN (
  SELECT CONCAT(
    'Revenue Summary\n',
    'Segment: ', CASE WHEN segment = -1 THEN 'ALL' ELSE CAST(segment AS STRING) END,
    ' | State: ', state_filter, '\n',
    'Customers: ', CAST(COUNT(*) AS STRING), '\n',
    'Total revenue: $', CAST(FORMAT_NUMBER(SUM(monetary), 2) AS STRING), '\n',
    'Avg revenue/customer: $', CAST(FORMAT_NUMBER(AVG(monetary), 2) AS STRING), '\n',
    'Total orders: ', CAST(CAST(SUM(num_orders) AS BIGINT) AS STRING), '\n',
    'Avg orders/customer: ', CAST(ROUND(AVG(num_orders), 1) AS STRING), '\n',
    'Customers with orders: ', CAST(SUM(CASE WHEN has_orders = 1 THEN 1 ELSE 0 END) AS STRING),
    ' (', CAST(ROUND(100.0 * SUM(CASE WHEN has_orders = 1 THEN 1 ELSE 0 END) / COUNT(*), 1) AS STRING), '%)'
  )
  FROM workspace.default.gold_customer_360
  WHERE (segment = -1 OR loyalty_segment = segment)
    AND (state_filter = 'ALL' OR state = state_filter)
);

-- Szybki test: przychód VIPów w NY
SELECT workspace.default.get_revenue_summary(3, 'NY') AS revenue_report;

revenue_report
"Revenue Summary Segment: 3 | State: NY Customers: 1140 Total revenue: $1,244,468.00 Avg revenue/customer: $1,091.64 Total orders: 480 Avg orders/customer: 0.4 Customers with orders: 153 (13.4%)"


In [0]:
%python
# Tool calling: LLM sam decyduje którą funkcję wywołać na podstawie pytania użytkownika
import json
from openai import OpenAI
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
client = OpenAI(
    api_key=w.config.authenticate()["Authorization"].split(" ", 1)[1],
    base_url=f"{w.config.host}/serving-endpoints",
)
LLM = "databricks-meta-llama-3-3-70b-instruct"

# 1. Definiujemy narzędzie (opis UC Function w formacie OpenAI tools)
tools = [{
    "type": "function",
    "function": {
        "name": "get_revenue_summary",
        "description": "Returns revenue summary from gold_customer_360: total revenue, avg per customer, count, orders. Use for questions about revenue by segment and geography.",
        "parameters": {
            "type": "object",
            "properties": {
                "segment": {"type": "integer", "description": "Loyalty segment 0=new, 1=occasional, 2=regular, 3=VIP, -1=all"},
                "state_filter": {"type": "string", "description": "US state abbreviation (NY, CA...) or ALL"}
            },
            "required": ["segment", "state_filter"]
        }
    }
}]

# 2. Pytamy LLM — model WYBIERA narzędzie i parametry
user_question = "Jaki jest łączny przychód od klientów VIP w stanie Nowy Jork?"
print(f"❓ {user_question}\n")

response = client.chat.completions.create(
    model=LLM,
    messages=[
        {"role": "system", "content": "You are a retail analytics assistant. Use tools to answer revenue questions. Always call the tool — never guess numbers."},
        {"role": "user", "content": user_question},
    ],
    tools=tools,
    tool_choice="auto",
)

msg = response.choices[0].message
if msg.tool_calls:
    tc = msg.tool_calls[0]
    args = json.loads(tc.function.arguments)
    print(f"🛠️  LLM wybrał narzędzie: {tc.function.name}")
    print(f"   Parametry: segment={args.get('segment')}, state_filter={args.get('state_filter')}")

    # 3. Wykonujemy UC Function z parametrami wybranymi przez LLM (bezpiecznie, przez args={})
    result = spark.sql(
        "SELECT workspace.default.get_revenue_summary(:segment, :state_filter)",
        args={"segment": args["segment"], "state_filter": args["state_filter"]},
    ).first()[0]
    print(f"\n📊 Wynik z UC Function:\n{result}")

    # 4. Odsyłamy wynik do LLM — model formatuje odpowiedź
    final = client.chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": "You are a retail analytics assistant. Answer in Polish, concisely."},
            {"role": "user", "content": user_question},
            {"role": "assistant", "content": None, "tool_calls": [{"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}]},
            {"role": "tool", "content": result, "tool_call_id": tc.id},
        ],
    )
    print(f"\n💬 Odpowiedź agenta:\n{final.choices[0].message.content}")
else:
    print("LLM nie wywołał narzędzia:", msg.content)

print("\n💡 To jest mechanizm tool calling — LLM sam zdecydował, że potrzebuje get_revenue_summary")
print("   i sam dobrał parametry (segment=3 → VIP, state_filter=NY).")
print("   W Warsztacie 4 zbudujemy pełnego agenta z wieloma narzędziami + guardrails + MCP.")

❓ Jaki jest łączny przychód od klientów VIP w stanie Nowy Jork?

🛠️  LLM wybrał narzędzie: get_revenue_summary
   Parametry: segment=3, state_filter=NY

📊 Wynik z UC Function:
Revenue Summary
Segment: 3 | State: NY
Customers: 1140
Total revenue: $1,244,468.00
Avg revenue/customer: $1,091.64
Total orders: 480
Avg orders/customer: 0.4
Customers with orders: 153 (13.4%)

💬 Odpowiedź agenta:
Łączny przychód od klientów VIP w stanie Nowy Jork wynosi $1,244,468.00.

💡 To jest mechanizm tool calling — LLM sam zdecydował, że potrzebuje get_revenue_summary
   i sam dobrał parametry (segment=3 → VIP, state_filter=NY).
   W Warsztacie 4 zbudujemy pełnego agenta z wieloma narzędziami + guardrails + MCP.


## Sekcja 6: Machine Learning z MLflow
**Funkcjonalności:** scikit-learn, MLflow autologging, tracking eksperymentów, porównanie runów

MLflow to open-source platforma do zarządzania cyklem życia modeli ML.
W Databricks jest w pełni zintegrowana — automatycznie loguje parametry, metryki i artefakty.

**Nasz cel:** Przewidzieć `loyalty_segment` klienta (0–3) na podstawie jego cech zakupowych.
To **klasyfikacja wieloklasowa** — model nauczy się, którzy klienci są VIP, a którzy
mogą odejść. W części 2 warsztatu zewaluujemy chatbota, który korzysta z tych danych.

In [0]:
%python
import mlflow
import mlflow.sklearn
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd
import numpy as np

# Przygotowanie danych do modelu klasyfikacji loyalty_segment
GOLD_TABLE = "workspace.default.gold_customer_360"

feature_cols = [
    "units_purchased", "recency_days", "frequency", "num_orders",
    "monetary", "avg_item_value", "promo_orders", "has_orders", "promo_ratio",
    "lat", "lon"
]
target_col = "loyalty_segment"

pdf = spark.table(GOLD_TABLE).select(feature_cols + [target_col]).toPandas()
pdf = pdf.dropna()

X = pdf[feature_cols]
y = pdf[target_col]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set: {len(X_train)} rows")
print(f"Test set: {len(X_test)} rows")
print(f"Features: {feature_cols}")
print(f"Target: {target_col} (classes: {sorted(y.unique())})")
print(f"Class distribution:\n{y.value_counts().sort_index().to_string()}")

Training set: 23050 rows
Test set: 5763 rows
Features: ['units_purchased', 'recency_days', 'frequency', 'num_orders', 'monetary', 'avg_item_value', 'promo_orders', 'has_orders', 'promo_ratio', 'lat', 'lon']
Target: loyalty_segment (classes: [np.int64(0), np.int64(1), np.int64(2), np.int64(3)])
Class distribution:
loyalty_segment
0    11097
1     3883
2     4292
3     9541


In [0]:
%python
# Model 1: Gradient Boosting Classifier
mlflow.sklearn.autolog()  # Automatyczne logowanie wszystkiego!

with mlflow.start_run(run_name="GBClassifier_loyalty"):
    model_gb = GradientBoostingClassifier(
        n_estimators=200, 
        max_depth=6, 
        learning_rate=0.1,
        random_state=42
    )
    model_gb.fit(X_train, y_train)
    
    preds_gb = model_gb.predict(X_test)
    
    acc_gb = accuracy_score(y_test, preds_gb)
    f1_gb = f1_score(y_test, preds_gb, average='weighted')
    
    print(f"Gradient Boosting Classifier:")
    print(f"   Accuracy: {acc_gb:.4f}")
    print(f"   F1 (weighted): {f1_gb:.4f}")
    print(f"\n{classification_report(y_test, preds_gb)}")

2026/09/08 10:59:46 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/09/08 10:59:57 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/ml

Gradient Boosting Classifier:
   Accuracy: 1.0000
   F1 (weighted): 1.0000

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      2220
           1       1.00      1.00      1.00       777
           2       1.00      1.00      1.00       858
           3       1.00      1.00      1.00      1908

    accuracy                           1.00      5763
   macro avg       1.00      1.00      1.00      5763
weighted avg       1.00      1.00      1.00      5763



In [0]:
%python
# Model 2: Random Forest Classifier (do porównania w MLflow UI)

with mlflow.start_run(run_name="RFClassifier_loyalty"):
    model_rf = RandomForestClassifier(
        n_estimators=200, 
        max_depth=10, 
        random_state=42
    )
    model_rf.fit(X_train, y_train)
    
    preds_rf = model_rf.predict(X_test)
    
    acc_rf = accuracy_score(y_test, preds_rf)
    f1_rf = f1_score(y_test, preds_rf, average='weighted')
    
    print(f"Random Forest Classifier:")
    print(f"   Accuracy: {acc_rf:.4f}")
    print(f"   F1 (weighted): {f1_rf:.4f}")

better = 'Gradient Boosting' if f1_gb > f1_rf else 'Random Forest'
print(f"\nLepszy model: {better} (F1: {max(f1_gb, f1_rf):.4f})")

# Tip: Kliknij "Experiments" w sidebar → porównaj oba runy wizualnie!

2026/09/08 11:00:12 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/09/08 11:00:15 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.12/site-packages/ml

Random Forest Classifier:
   Accuracy: 1.0000
   F1 (weighted): 1.0000

Lepszy model: Random Forest (F1: 1.0000)


## Sekcja 7: Model Registry w Unity Catalog
**Funkcjonalności:** Rejestracja modelu, wersjonowanie, governance modeli

Najlepszy model (klasyfikator loyalty_segment) rejestrujemy w Unity Catalog —
dzięki temu jest zarządzany, wersjonowany i dostępny dla całej organizacji.
Model trafi też do części 2 warsztatu, gdzie będziemy monitorować jego wyniki.

**Alias `@champion`.** Numer wersji (`/1`, `/2`…) zmienia się przy każdym retrainingu. Zamiast wpisywać go na sztywno w batch inference, monitoringu (WS2) czy serving (WS4), nadajemy wersji **alias** `champion`. Konsumenci odwołują się do `models:/nazwa@champion`, a promocja nowej wersji to jedna operacja na aliasie — bez zmiany kodu konsumentów.

> WS2 (komórka *Jakość modelu ML z WS1*) ładuje właśnie `@champion` — bez tej komórki tamten test kończył się błędem „model nie jest dostępny”.

In [0]:
%python

# Rejestracja najlepszego modelu w Unity Catalog + alias @champion
from mlflow import MlflowClient

best_run = mlflow.last_active_run()
model_name = "workspace.default.loyalty_segment_classifier"

mlflow.set_registry_uri("databricks-uc")
registered_model = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_name
)

# Alias = ruchomy wskaźnik na wersję. Konsumenci (WS2 monitoring, WS4 serving) używają models:/...@champion
client = MlflowClient()
client.set_registered_model_alias(name=model_name, alias="champion", version=registered_model.version)
champion = client.get_model_version_by_alias(name=model_name, alias="champion")

print(f"Model zarejestrowany: {model_name}")
print(f"   Wersja: {registered_model.version}")
print(f"   Run ID: {best_run.info.run_id}")
print(f"   Alias:  @champion -> wersja {champion.version}")
print(f"   URI dla konsumentów: models:/{model_name}@champion")

Registered model 'workspace.default.loyalty_segment_classifier' already exists. Creating a new version of this model...
2026/09/08 11:00:23 WARNING mlflow.tracking._model_registry.fluent: Run with id 620ded0b4045414fb3f6a56040436aaf has no artifacts at artifact path 'model', registering model based on models:/m-77e012747e1849c4b384520e26cca9a5 instead


Uploading artifacts:   0%|          | 0/10 [00:00<?, ?it/s]

🔗 Created version '4' of model 'workspace.default.loyalty_segment_classifier': https://adb-7405615837166522.2.azuredatabricks.net/explore/data/models/sandbox/testy/loyalty_segment_classifier/version/4?o=7405615837166522


Model zarejestrowany: workspace.default.loyalty_segment_classifier
   Wersja: 4
   Run ID: 620ded0b4045414fb3f6a56040436aaf
   Alias:  @champion -> wersja 4
   URI dla konsumentów: models:/workspace.default.loyalty_segment_classifier@champion


## Sekcja 8: Predykcja na nowych danych (Batch Inference)
**Funkcjonalności:** Model z UC Registry, skalowalna inferencja, porównanie predykcji

Załadowany model działa jako zwykła funkcja Python — możemy go zastosować
na całej tabeli klientów i porównać predykcje z rzeczywistymi segmentami.

In [0]:
%python
# Załadowanie modelu z Unity Catalog Model Registry
import os
os.environ["MLFLOW_OPENAI_RETRIES"] = "0"

# Ładujemy przez alias, nie numer wersji — po retrainingu wystarczy przepiąć alias
loaded_model = mlflow.pyfunc.load_model(f"models:/{model_name}@champion")

# Batch inference na całej tabeli Gold
GOLD_TABLE = "workspace.default.gold_customer_360"

inference_pdf = spark.table(GOLD_TABLE).select(feature_cols + [target_col, "customer_name", "state"]).toPandas().dropna()

inference_pdf["predicted_segment"] = loaded_model.predict(inference_pdf[feature_cols])
inference_pdf["correct"] = (inference_pdf["loyalty_segment"] == inference_pdf["predicted_segment"]).astype(int)

# Podsumowanie
acc = inference_pdf["correct"].mean()
print(f"Batch inference na {len(inference_pdf):,} klientach")
print(f"Accuracy: {acc:.4f}")
print(f"\nPredykcje per segment (pierwsze 15):")

# Konwersja na Spark DataFrame do display
predictions = spark.createDataFrame(
    inference_pdf[["customer_name", "state", "loyalty_segment", "predicted_segment", "correct"]].head(15)
)
display(predictions)

Batch inference na 28,813 klientach
Accuracy: 1.0000

Predykcje per segment (pierwsze 15):


customer_name,state,loyalty_segment,predicted_segment,correct
"SMITH, SHIRLEY",IN,3,3,1
"STEPHENS, GERALDINE M",OR,3,3,1
"GUZMAN, CARMEN",VA,0,0,1
"HASSETT, PATRICK J",WI,1,1,1
"HENTZ, DIANA L",OH,0,0,1
"TIRADO, MARCO A",NY,3,3,1
"SKORA, BRIAN S",MI,1,1,1
"SLAWEK, DEAN J",PA,3,3,1
"REAVES, LIONEL C",VA,2,2,1
"BONGIOVANNI, KELLY M",IN,2,2,1


## Sekcja 8b: AutoML — automatyczny dobór modelu
**Funkcjonalności:** Databricks AutoML, automatyczne porównanie modeli, feature importance

W sekcjach 6–8 ręcznie trenowaliśmy Gradient Boosting i Random Forest do klasyfikacji
`loyalty_segment`. **AutoML** robi to automatycznie:
- Testuje wiele algorytmów (XGBoost, LightGBM, RandomForest, Logistic...)
- Optymalizuje hiperparametry
- Generuje gotowy notebook z najlepszym modelem
- Loguje wszystko do MLflow

> **Wymaga klastra z ML Runtime** (np. 16.x ML). Na Serverless compute AutoML nie jest dostępny.
> Przed uruchomieniem tej sekcji przełącz compute na klaster z ML Runtime.

> AutoML to świetny punkt startowy — nawet jeśli potem dostrajasz model ręcznie.

In [0]:
%python
try:
    from databricks import automl
except ImportError:
    print("AutoML wymaga klastra z Databricks ML Runtime (np. 16.x ML).")
    print("   Na Serverless compute ten moduł nie jest dostępny.")
    print("\nAby uruchomić AutoML:")
    print("   1. Utwórz klaster z ML Runtime (np. 16.1 ML)")
    print("   2. Podłącz notebook do tego klastra")
    print("   3. Uruchom tę komórkę ponownie")
    automl = None

if automl:
    # Przygotowanie danych do AutoML (używamy tej samej tabeli Gold)
    automl_df = spark.table("workspace.default.gold_customer_360").select(
        "loyalty_segment",
        "units_purchased", "recency_days", "frequency", "num_orders",
        "monetary", "avg_item_value", "promo_orders", "has_orders", "promo_ratio",
        "lat", "lon"
    ).dropna()

    # Uruchomienie AutoML (klasyfikacja — predykcja loyalty_segment)
    summary = automl.classify(
        dataset=automl_df,
        target_col="loyalty_segment",
        primary_metric="f1",
        timeout_minutes=5,       # Limit czasu (na warsztaty: 5 min)
        max_trials=10,           # Max liczba modeli do przetestowania
    )

    # Wyniki
    metrics = summary.best_trial.metrics
    rmse_key = next((k for k in metrics if 'rmse' in k.lower()), None)
    r2_key = next((k for k in metrics if 'r2' in k.lower()), None)

    print(f"\nAutoML zakończony!")
    print(f"   Najlepszy model: {summary.best_trial.model_description}")
    if rmse_key:
        print(f"   RMSE: {metrics[rmse_key]:.2f}")
    if r2_key:
        print(f"   R²:   {metrics[r2_key]:.4f}")
    print(f"   MLflow Experiment: {summary.experiment.name}")
    print(f"\nNotebook z najlepszym modelem: {summary.best_trial.notebook_path}")
    print(f"\nWszystkie metryki: {list(metrics.keys())}")

AutoML wymaga klastra z Databricks ML Runtime (np. 16.x ML).
   Na Serverless compute ten moduł nie jest dostępny.

Aby uruchomić AutoML:
   1. Utwórz klaster z ML Runtime (np. 16.1 ML)
   2. Podłącz notebook do tego klastra
   3. Uruchom tę komórkę ponownie


In [0]:
%python
# --- Użycie najlepszego modelu z AutoML ---
# AutoML automatycznie loguje modele do MLflow — ładujemy najlepszy i porównujemy z ręcznym

if automl and summary:
    import mlflow
    import pandas as pd

    # Załaduj najlepszy model z AutoML
    best_model_uri = f"runs:/{summary.best_trial.mlflow_run_id}/model"
    automl_model = mlflow.pyfunc.load_model(best_model_uri)
    print(f"Załadowano najlepszy model AutoML: {summary.best_trial.model_description}")
    print(f"   Run ID: {summary.best_trial.mlflow_run_id}")

    # Predykcja na tych samych danych testowych co ręczny model
    automl_predictions = automl_model.predict(X_test)

    # Porównanie: AutoML vs ręczny Gradient Boosting
    from sklearn.metrics import accuracy_score, f1_score

    automl_acc = accuracy_score(y_test, automl_predictions)
    automl_f1 = f1_score(y_test, automl_predictions, average='weighted')

    # Metryki ręcznego modelu (jeśli dostępny)
    manual_preds = model_gb.predict(X_test) if 'model_gb' in dir() else None

    print(f"\nPorównanie modeli na zbiorze testowym:")
    print(f"{'Metryka':<12} {'AutoML':<20} {'Ręczny (GB)':<20}")
    print(f"{'-'*52}")
    if manual_preds is not None:
        manual_acc = accuracy_score(y_test, manual_preds)
        manual_f1 = f1_score(y_test, manual_preds, average='weighted')
        print(f"{'Accuracy':<12} {automl_acc:<20.4f} {manual_acc:<20.4f}")
        print(f"{'F1':<12} {automl_f1:<20.4f} {manual_f1:<20.4f}")
        winner = "AutoML" if automl_f1 > manual_f1 else "Ręczny GB"
        print(f"\nLepszy model: {winner}")
    else:
        print(f"{'Accuracy':<12} {automl_acc:.4f}")
        print(f"{'F1':<12} {automl_f1:.4f}")
else:
    print("AutoML nie został uruchomiony — uruchom najpierw komórkę powyżej.")

AutoML nie został uruchomiony — uruchom najpierw komórkę powyżej.


## Sekcja 9: Prognoza dziennego przychodu (Random Forest)
**Funkcjonalności:** RandomForestRegressor, cechy czasowe, prognoza szeregów czasowych

Agregujemy zamówienia z Marketplace do dziennych przychodów i prognozujemy
następne 14 dni za pomocą **Random Forest** — tego samego algorytmu co w Sekcji 6.

- **Spójność** — uczestnicy już znają Random Forest z sekcji 6
- **Cechy czasowe** — dzień tygodnia, dzień miesiąca, trend
- **Przedziały ufności** — via wariancję predykcji poszczególnych drzew

> **Tip:** Po uruchomieniu kliknij "+" przy wynikach, wybierz wykres liniowy
> (X=date, Y=predicted_revenue) — zobaczysz prognozę z przedziałami ufności.

In [0]:
%python
# --- Prognoza dziennego przychodu: Random Forest (14 dni) ---
# Ten sam algorytm co w Sekcji 6 — uczestnicy już go znają!

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from pyspark.sql import functions as F

# Krok 1: Agregacja dziennego przychodu z zamówień
catalog = "databricks_simulated_retail_customer_data"
orders = spark.table(f"{catalog}.v01.sales_orders")

product_schema = "ARRAY<STRUCT<curr:STRING, id:STRING, name:STRING, price:STRING, qty:STRING, unit:STRING>>"

daily_revenue = (
    orders
    .withColumn("order_date", F.from_unixtime("order_datetime").cast("date"))
    .withColumn("products", F.from_json("ordered_products", product_schema))
    .withColumn("product", F.explode("products"))
    .withColumn("item_revenue", F.col("product.price").cast("double") * F.col("product.qty").cast("int"))
    .groupBy("order_date")
    .agg(F.round(F.sum("item_revenue"), 2).alias("daily_revenue"),
         F.count("*").alias("num_items"))
    .toPandas()
)
daily_revenue["order_date"] = pd.to_datetime(daily_revenue["order_date"])
daily_revenue = daily_revenue.sort_values("order_date").set_index("order_date").asfreq("D").fillna(0)

def make_time_features(dates):
    """Tworzy cechy czasowe z dat."""
    return pd.DataFrame({
        "day_of_week": dates.dayofweek,
        "day_of_month": dates.day,
        "is_weekend": (dates.dayofweek >= 5).astype(int),
        "trend": np.arange(len(dates)),
    }, index=dates)

# Krok 2: Trening modelu
X_train = make_time_features(daily_revenue.index)
y_train = daily_revenue["daily_revenue"].values

model = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)
model.fit(X_train, y_train)
r2 = model.score(X_train, y_train)
print(f"R² na danych treningowych: {r2:.3f}")

# Krok 3: Prognoza na 14 dni
horizon = 14
future_dates = pd.date_range(
    daily_revenue.index.max() + pd.Timedelta(days=1), periods=horizon, freq="D"
)
X_future = make_time_features(future_dates)
X_future["trend"] = np.arange(len(X_train), len(X_train) + horizon)

forecast = model.predict(X_future)

# Przedziały ufności z poszczególnych drzew
tree_preds = np.array([tree.predict(X_future.values) for tree in model.estimators_])
lower = np.percentile(tree_preds, 5, axis=0)
upper = np.percentile(tree_preds, 95, axis=0)

df_forecast = pd.DataFrame({
    "date": future_dates,
    "predicted_revenue": forecast.round(2),
    "lower_bound": lower.round(2),
    "upper_bound": upper.round(2),
})

print(f"\nPrognoza na {horizon} dni: {df_forecast['date'].min().date()} — {df_forecast['date'].max().date()}")
print(f"Średni prognozowany przychód: ${df_forecast['predicted_revenue'].mean():,.0f}/dzień")

# Feature importance
print(f"\nFeature importance:")
for feat, imp in sorted(zip(X_train.columns, model.feature_importances_), key=lambda x: -x[1]):
    print(f"   {feat:<15} {imp:.3f}")

# Tip: kliknij "+" → wykres liniowy, X=date, Y=predicted_revenue
display(spark.createDataFrame(df_forecast))

R² na danych treningowych: 0.934

Prognoza na 14 dni: 2019-11-15 — 2019-11-28
Średni prognozowany przychód: $92,570/dzień

Feature importance:
   day_of_week     0.628
   trend           0.156
   is_weekend      0.112
   day_of_month    0.104


date,predicted_revenue,lower_bound,upper_bound
2019-11-15T00:00:00.000Z,63017.25,44905.0,125633.0
2019-11-16T00:00:00.000Z,51577.74,15506.0,66553.0
2019-11-17T00:00:00.000Z,53045.07,21801.0,66553.0
2019-11-18T00:00:00.000Z,156796.95,130887.4,203833.0
2019-11-19T00:00:00.000Z,172744.71,131302.0,203833.0
2019-11-20T00:00:00.000Z,74053.91,44905.0,151204.0
2019-11-21T00:00:00.000Z,77437.38,44905.0,153452.6
2019-11-22T00:00:00.000Z,71099.21,44905.0,128727.95
2019-11-23T00:00:00.000Z,49415.08,35837.0,66553.0
2019-11-24T00:00:00.000Z,51476.02,21801.0,66553.0


Databricks visualization. Run in Databricks to view.

## Sekcja 10: Udostępnianie wyników — Dashboard + Genie Space
**Funkcjonalności:** Databricks SDK, Lakeview Dashboard API, Genie Spaces API

Na koniec pokazujemy jak **programowo** udostępnić wyniki biznesowi:
- **AI/BI Dashboard** — interaktywny dashboard z danych `gold_customer_360`
- **Genie Space** — konwersacyjna eksploracja danych w języku naturalnym

Dzięki temu cały pipeline (od Marketplace do dashboardu) można zautomatyzować!

In [0]:
%python
import json
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
host = w.config.host
headers = w.config.authenticate()

dashboard_name = "Retail Customer Intelligence — Workshop Dashboard"

# 1) Datasety SQL
raw_datasets = [
    {
        "name": "kpi_metrics",
        "displayName": "KPI Metrics",
        "query": "SELECT COUNT(*) as total_customers, COUNT(DISTINCT state) as states, ROUND(AVG(monetary), 2) as avg_monetary, ROUND(AVG(units_purchased), 1) as avg_units FROM workspace.default.gold_customer_360",
    },
    {
        "name": "sales_by_category_region",
        "displayName": "Sprzedaż per kategoria i region",
        "query": "SELECT loyalty_segment, state, COUNT(*) as customers, ROUND(AVG(monetary), 0) as avg_revenue FROM workspace.default.gold_customer_360 WHERE state IN ('NY','CA','FL','OH','MA') GROUP BY loyalty_segment, state ORDER BY customers DESC",
    },
    {
        "name": "monthly_trend",
        "displayName": "Trend miesięczny",
        "query": "SELECT DATE_TRUNC('month', `Date`) as month, `Category`, SUM(`Units Sold`) as total_sold FROM workspace.default.gold_customer_360 GROUP BY 1, 2 ORDER BY 1",
    },
    {
        "name": "store_ranking",
        "displayName": "Ranking sklepów",
        "query": "SELECT `Store ID`, SUM(`Units Sold`) as total_sold FROM workspace.default.gold_customer_360 WHERE `Date` >= '2023-01-01' GROUP BY `Store ID` ORDER BY total_sold DESC",
    },
    {
        "name": "weather_impact",
        "displayName": "Wpływ pogody",
        "query": "SELECT `Weather Condition`, `Category`, ROUND(AVG(`Units Sold`), 0) as avg_sales FROM workspace.default.gold_customer_360 GROUP BY 1, 2",
    },
]

datasets = [
    {
        "name": d["name"],
        "displayName": d["displayName"],
        "queryLines": [d["query"]],
    }
    for d in raw_datasets
]

# 2) Widget definitions w formacie akceptowanym przez Lakeview API
widgets = [
    {
        "name": "kpi_units",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "kpi_metrics",
                "fields": [{"name": "total_units_sold", "expression": "`total_units_sold`"}],
                "disaggregated": True,
            },
        }],
        "spec": {
            "version": 2,
            "frame": {"title": "Total Units Sold", "showTitle": True},
            "widgetType": "counter",
            "encodings": {"value": {"fieldName": "total_units_sold", "rowNumber": 0}},
            "data": {"queryName": "main_query"},
        },
    },
    {
        "name": "kpi_stores",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "kpi_metrics",
                "fields": [{"name": "stores", "expression": "`stores`"}],
                "disaggregated": True,
            },
        }],
        "spec": {
            "version": 2,
            "frame": {"title": "Stores", "showTitle": True},
            "widgetType": "counter",
            "encodings": {"value": {"fieldName": "stores", "rowNumber": 0}},
            "data": {"queryName": "main_query"},
        },
    },
    {
        "name": "kpi_products",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "kpi_metrics",
                "fields": [{"name": "products", "expression": "`products`"}],
                "disaggregated": True,
            },
        }],
        "spec": {
            "version": 2,
            "frame": {"title": "Products", "showTitle": True},
            "widgetType": "counter",
            "encodings": {"value": {"fieldName": "products", "rowNumber": 0}},
            "data": {"queryName": "main_query"},
        },
    },
    {
        "name": "sales_bar",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "sales_by_category_region",
                "fields": [
                    {"name": "Region", "expression": "`Region`"},
                    {"name": "Category", "expression": "`Category`"},
                    {"name": "sum(total_sold)", "expression": "SUM(`total_sold`)"},
                ],
                "disaggregated": False,
            },
        }],
        "spec": {
            "version": 3,
            "frame": {"title": "Sprzedaż per kategoria i region", "showTitle": True},
            "widgetType": "bar",
            "encodings": {
                "x": {"fieldName": "Category", "displayName": "Category", "scale": {"type": "categorical"}},
                "y": {"fieldName": "sum(total_sold)", "displayName": "total_sold", "scale": {"type": "quantitative"}},
                "color": {"fieldName": "Region", "displayName": "Region", "scale": {"type": "categorical"}},
            },
            "data": {"queryName": "main_query"},
        },
    },
    {
        "name": "trend_line",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "monthly_trend",
                "fields": [
                    {"name": "Category", "expression": "`Category`"},
                    {"name": "month", "expression": "`month`"},
                    {"name": "sum(total_sold)", "expression": "SUM(`total_sold`)"},
                ],
                "disaggregated": False,
            },
        }],
        "spec": {
            "version": 3,
            "frame": {"title": "Trend miesięczny sprzedaży", "showTitle": True},
            "widgetType": "line",
            "encodings": {
                "x": {"fieldName": "month", "displayName": "month", "scale": {"type": "temporal"}},
                "y": {"fieldName": "sum(total_sold)", "displayName": "total_sold", "scale": {"type": "quantitative"}},
                "color": {"fieldName": "Category", "displayName": "Category", "scale": {"type": "categorical"}},
            },
            "data": {"queryName": "main_query"},
        },
    },
    {
        "name": "store_bar",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "store_ranking",
                "fields": [
                    {"name": "Store ID", "expression": "`Store ID`"},
                    {"name": "sum(total_sold)", "expression": "SUM(`total_sold`)"},
                ],
                "disaggregated": False,
            },
        }],
        "spec": {
            "version": 3,
            "frame": {"title": "Ranking sklepów (od 2023)", "showTitle": True},
            "widgetType": "bar",
            "encodings": {
                "x": {"fieldName": "Store ID", "displayName": "Store ID", "scale": {"type": "categorical"}},
                "y": {"fieldName": "sum(total_sold)", "displayName": "total_sold", "scale": {"type": "quantitative"}},
                "color": {"fieldName": "Store ID", "displayName": "Store ID", "scale": {"type": "categorical"}},
            },
            "data": {"queryName": "main_query"},
        },
    },
    {
        "name": "weather_pivot",
        "queries": [{
            "name": "main_query",
            "query": {
                "datasetName": "weather_impact",
                "fields": [
                    {"name": "Weather Condition", "expression": "`Weather Condition`"},
                    {"name": "Category", "expression": "`Category`"},
                    {"name": "sum(avg_sales)", "expression": "SUM(`avg_sales`)"},
                ],
                "cubeGroupingSets": {"sets": [{"fieldNames": ["Weather Condition"]}, {"fieldNames": ["Category"]}]},
                "disaggregated": False,
                "orders": [
                    {"direction": "ASC", "expression": "`Weather Condition`"},
                    {"direction": "ASC", "expression": "`Category`"},
                ],
            },
        }],
        "spec": {
            "version": 3,
            "frame": {"title": "Wpływ pogody na sprzedaż", "showTitle": True},
            "widgetType": "pivot",
            "encodings": {
                "rows": [{"fieldName": "Weather Condition", "scale": {"type": "categorical"}}],
                "columns": [{"fieldName": "Category", "scale": {"type": "categorical"}}],
                "cell": {
                    "type": "single-cell",
                    "fieldName": "sum(avg_sales)",
                    "style": {
                        "type": "color-scale",
                        "backgroundColor": {"scale": {"type": "quantitative"}},
                    },
                },
            },
            "data": {"queryName": "main_query"},
        },
    },
]

# 3) Layout strony; widgety są osadzone bezpośrednio w pages[].layout
layout = [
    {"widget": widgets[0], "position": {"x": 0, "y": 0, "width": 2, "height": 2}},
    {"widget": widgets[1], "position": {"x": 2, "y": 0, "width": 2, "height": 2}},
    {"widget": widgets[2], "position": {"x": 4, "y": 0, "width": 2, "height": 2}},
    {"widget": widgets[3], "position": {"x": 0, "y": 2, "width": 3, "height": 4}},
    {"widget": widgets[4], "position": {"x": 3, "y": 2, "width": 3, "height": 4}},
    {"widget": widgets[5], "position": {"x": 0, "y": 6, "width": 3, "height": 4}},
    {"widget": widgets[6], "position": {"x": 3, "y": 6, "width": 3, "height": 4}},
]

serialized_dashboard = json.dumps({
    "datasets": datasets,
    "pages": [{
        "name": "overview",
        "displayName": "Retail Forecasting Overview",
        "layout": layout,
        "pageType": "PAGE_TYPE_CANVAS",
        "layoutVersion": "GRID_V1",
    }],
    "uiSettings": {
        "theme": {"widgetHeaderAlignment": "ALIGNMENT_UNSPECIFIED"},
        "applyModeEnabled": False,
    },
})

# 4) PATCH jeśli istnieje, POST jeśli nie
list_resp = requests.get(f"{host}/api/2.0/lakeview/dashboards", headers=headers)
list_resp.raise_for_status()
existing = next((d for d in list_resp.json().get("dashboards", []) if d["display_name"] == dashboard_name), None)

if existing:
    dashboard_id = existing["dashboard_id"]
    resp = requests.patch(
        f"{host}/api/2.0/lakeview/dashboards/{dashboard_id}",
        headers=headers,
        json={"serialized_dashboard": serialized_dashboard},
    )
    resp.raise_for_status()
    action = "zaktualizowany"
else:
    resp = requests.post(
        f"{host}/api/2.0/lakeview/dashboards",
        headers=headers,
        json={"display_name": dashboard_name, "serialized_dashboard": serialized_dashboard},
    )
    resp.raise_for_status()
    dashboard_id = resp.json()["dashboard_id"]
    action = "utworzony"

# 5) Podsumowanie
print(f"Dashboard {action}: {dashboard_name}")
print(f"   ID: {dashboard_id}")
print(f"   URL: {host}/sql/dashboardsv3/{dashboard_id}")
print(f"\nWidgety ({len(widgets)}):")
for w in widgets:
    print(f"   • [{w['spec']['widgetType']}] {w['spec']['frame']['title']}")
print("\nDashboard jest gotowy z wizualizacjami od razu po utworzeniu/aktualizacji.")

Dashboard zaktualizowany: Retail Customer Intelligence — Workshop Dashboard
   ID: 01f1a5f8979f17c58f4f5de8959aa770
   URL: https://adb-7405615837166522.2.azuredatabricks.net/sql/dashboardsv3/01f1a5f8979f17c58f4f5de8959aa770

Widgety (7):
   • [counter] Total Units Sold
   • [counter] Stores
   • [counter] Products
   • [bar] Sprzedaż per kategoria i region
   • [line] Trend miesięczny sprzedaży
   • [bar] Ranking sklepów (od 2023)
   • [pivot] Wpływ pogody na sprzedaż

Dashboard jest gotowy z wizualizacjami od razu po utworzeniu/aktualizacji.


In [0]:
%python
# --- Tworzenie Genie Space (REST API) ---
# Genie pozwala użytkownikom biznesowym zadawać pytania w języku naturalnym

user_email = spark.sql("SELECT current_user()").collect()[0][0]

# Potrzebujemy warehouse_id — szukamy dostępnego SQL Warehouse
wh_response = requests.get(f"{host}/api/2.0/sql/warehouses", headers=headers)
warehouses = wh_response.json().get("warehouses", [])
running_wh = [wh for wh in warehouses if wh.get("state") == "RUNNING"]
warehouse_id = (running_wh[0] if running_wh else warehouses[0])["id"] if warehouses else ""
print(f"Używam SQL Warehouse: {warehouse_id}")

import uuid

def hex_id():
    return uuid.uuid4().hex

serialized_space = json.dumps({
    "version": 1,
    "config": {
        "sample_questions": [
            {"id": hex_id(), "question": ["Który sklep sprzedaje najwięcej w kategorii Electronics?"]},
            {"id": hex_id(), "question": ["Jak pogoda wpływa na sprzedaż Groceries?"]},
            {"id": hex_id(), "question": ["Pokaż trend sprzedaży dla regionu North w 2023"]},
            {"id": hex_id(), "question": ["Porównaj efektywność promocji między kategoriami"]}
        ]
    },
    "data_sources": {
        "tables": [
            {"identifier": "workspace.default.gold_customer_360",
             "description": ["Surowe dane sprzedażowe: 76K wierszy, 5 sklepów, 20 produktów, 5 kategorii, 4 regiony, daty 2022-2024"]},
            {"identifier": "workspace.default.gold_customer_360",
             "description": ["Dane z feature engineering: lag-i sprzedaży, średnie kroczace, cechy cenowe"]}
        ]
    },
    "instructions": {
        "text_instructions": [{
            "id": hex_id(),
            "content": ["Odpowiadaj po polsku. Kolumny mają spacje w nazwach — używaj backticków. Główna tabela to workspace.default.gold_customer_360 z kolumnami: Date, Store ID, Product ID, Category, Region, Units Sold, Price, Discount, Promotion, Weather Condition, Demand."]
        }]
    }
})

response = requests.post(
    f"{host}/api/2.0/genie/spaces",
    headers=headers,
    json={
        "title": "Retail Customer Intelligence Assistant",
        "description": "Asystent sprzedażowy — zadawaj pytania o sprzedaż, trendy, pogodę, promocje",
        "serialized_space": serialized_space,
        "warehouse_id": warehouse_id,
        "parent_path": f"/Workspace/Users/{user_email}"
    }
)
if response.status_code != 200:
    print(f"Status: {response.status_code}")
    print(f"   Response: {response.text[:500]}")
    raise Exception("Nie udało się utworzyć Genie Space")

genie = response.json()

print(f"Genie Space utworzony!")
print(f"   Tytuł: Retail Customer Intelligence Assistant")
print(f"   ID: {genie['space_id']}")
print(f"   URL: {host}/genie/rooms/{genie['space_id']}")
print(f"\nTabele w Genie Space:")
print(f"   • workspace.default.gold_customer_360 — surowe dane sprzedażowe")
print(f"   • workspace.default.gold_customer_360 — feature engineering")
print(f"\nPrzykładowe pytania:")
print(f"   • 'Który sklep sprzedaje najwięcej w kategorii Electronics?'")
print(f"   • 'Jak pogoda wpływa na sprzedaż Groceries?'")
print(f"   • 'Pokaż miesięczny trend sprzedaży dla regionu North'")

Używam SQL Warehouse: a440b61c4494ecec
Genie Space utworzony!
   Tytuł: Retail Customer Intelligence Assistant
   ID: 01f1ab748b8610b291c11d18998642a0
   URL: https://adb-7405615837166522.2.azuredatabricks.net/genie/rooms/01f1ab748b8610b291c11d18998642a0

Tabele w Genie Space:
   • workspace.default.gold_customer_360 — surowe dane sprzedażowe
   • workspace.default.gold_customer_360 — feature engineering

Przykładowe pytania:
   • 'Który sklep sprzedaje najwięcej w kategorii Electronics?'
   • 'Jak pogoda wpływa na sprzedaż Groceries?'
   • 'Pokaż miesięczny trend sprzedaży dla regionu North'


## Podsumowanie — co zbudowaliśmy w tym warsztacie

| # | Sekcja | Funkcjonalność Databricks | Język |
|---|---|---|---|
| 1 | Marketplace + Eksploracja | Databricks Marketplace, **przegląd licencji, DESCRIBE CATALOG (Delta Sharing)**, Unity Catalog, SQL w notebooku | SQL |
| 2 | Zaawansowana analityka | CTE, Window Functions, LAG, RANK, JOIN | SQL |
| 3 | AI Functions | ai_query(), ai_classify() — LLM w SQL | SQL |
| 3b | **Tool Calling** | UC Function `get_revenue_summary` + LLM tool calling (OpenAI API) | SQL + Python |
| 4 | Cleaning + Feature Eng. | PySpark, JSON parsing, JOIN, RFM features | Python |
| 5 | Delta Lake | saveAsTable, Time Travel, DESCRIBE HISTORY | Python + SQL |
| 6 | Machine Learning | scikit-learn (klasyfikacja), MLflow autologging | Python |
| 7 | Model Registry | Rejestracja w Unity Catalog **+ alias `@champion`** | Python |
| 8 | Batch Inference | Model z UC Registry **ładowany przez `@champion`**, predykcja loyalty_segment | Python |
| 8b | AutoML | Databricks AutoML, automatyczny dobór modelu | Python (ML Runtime) |
| 9 | Forecasting | Random Forest, cechy czasowe, prognoza przychodów | Python |
| 10 | Dashboard + Genie | Lakeview API, Genie Spaces API | Python (SDK) |

### Co dalej? → 3 kolejne warsztaty

**Warsztat 2: Guardrails, Monitoring i Ewaluacja** — zabezpieczamy i monitorujemy:
1. Guardrails LLM — system prompt + safety filter
2. Guardrails danych — column mask na `tax_id`, row filter per stan
3. Ewaluacja — `mlflow.genai.evaluate()` na Genie Space
4. Monitoring jakości — Lakehouse Monitoring (profil, dryf, dashboard)

**Warsztat 3: RAG i Knowledge Assistant** — budujemy chatbota:
1. Generowanie 10 artykułów PDF z `ai_query()` + `fpdf2`
2. Custom RAG z Vector Search (od zera)
3. Knowledge Assistant (Agent Bricks — managed RAG)
4. Porównanie Genie vs Custom RAG vs Knowledge Assistant

**Warsztat 4: Agent App** — pełna aplikacja:
1. UC Functions (SQL + Python UDF) na danych retail
2. LangChain Agent + MLflow Tracing
3. Model Serving endpoint
4. Databricks App (Gradio)

> Wszystko oparte na tabeli `workspace.default.gold_customer_360`, którą właśnie zbudowaliśmy.
- **Lakeflow Pipeline** - zautomatyzuj cały proces jako pipeline ETL
- **Delta Sharing** - udostępnij dane partnerom zewnętrznym
- **Alerts** - ustaw alert gdy sprzedaż spadnie poniżej progu
- **Scheduled Jobs** - uruchamiaj notebook automatycznie co tydzień
- **Model Serving** - udostępnij model jako REST API endpoint